# CryptoLabX — Assignment 10: MT19937 Stream Cipher & Predictable Token Attack
**Course**: Cryptography Laboratory  
**Group Number**: 17  
**Team Members**: Mukund Rakholiya (2024UCP1163), Lakshay Ahuja (2024UCP1157)  

---

## 1. Objective
Understand why a statistical PRNG is unsuitable for cryptographic applications by:
1. Implementing an 8-bit MT19937 toy stream cipher (symmetric XOR).
2. Generating a test message with a random prefix (5-20 chars) and known sequence of 14 'A's.
3. Encrypting with a weak 16-bit seed (0..65535).
4. Recovering the seed and full plaintext via exhaustive 65,536-seed brute force in under 1 second.
5. Analyzing predictable password reset tokens seeded with Unix timestamps.

In [ ]:
import os, sys
sys.path.append(os.path.abspath('..'))

from modern.mt19937_cipher import (
    MT19937StreamCipher,
    mt19937_cipher,
    verify_cipher_roundtrip,
    generate_plaintext,
    encrypt_with_16bit_seed,
    brute_force_16bit_seed,
    verify_recovered_result
)

## 2. Part A: Cipher Symmetry Verification
Verifying that encryption and decryption are perfectly symmetric.

In [ ]:
test_msg = b'CryptoLabX Assignment 10 Stream Cipher Roundtrip Test!'
cipher = MT19937StreamCipher(seed=42135)
ct = cipher.encrypt(test_msg)
pt = cipher.decrypt(ct)

print(f'Original        : {test_msg}')
print(f'Ciphertext (Hex): {ct.hex()}')
print(f'Decrypted       : {pt}')
print(f'Symmetry Match  : {pt == test_msg}')

## 3. Plaintext Generation & 16-Bit Seed Encryption
Generate plaintext with 5-20 random printable characters followed by 14 consecutive 'A's, then encrypt with a 16-bit seed:

In [ ]:
plaintext = generate_plaintext()
prefix = plaintext[:-14]
known_seq = plaintext[-14:]

ciphertext, secret_seed = encrypt_with_16bit_seed(plaintext)

print(f'Plaintext Length  : {len(plaintext)} bytes')
print(f"Random Prefix     : '{prefix.decode()}' ({len(prefix)} chars)")
print(f"Known Sequence    : '{known_seq.decode()}' (14 'A's)")
print(f'Full Plaintext    : {plaintext}')
print(f'Secret 16-bit Seed: {secret_seed} (Hex: 0x{secret_seed:04X})')
print(f'Ciphertext (Hex)  : {ciphertext.hex()}')

## 4. Exhaustive 65,536-Seed Brute-Force Key Recovery
Given only ciphertext and knowledge of 14 consecutive 'A's (location unknown), brute-force all 65,536 seeds:

In [ ]:
res = brute_force_16bit_seed(ciphertext, known_sequence=b'AAAAAAAAAAAAAA', scan_entire_space=True)

print(f'Seeds Tested        : {res.seeds_tested:,} / {res.total_search_space:,}')
print(f'Elapsed Time        : {res.elapsed_time*1000:.2f} ms ({res.elapsed_time:.4f} s)')
print(f'Throughput          : {res.seeds_per_second:,.0f} seeds/second')
print(f'Recovered Seed      : {res.recovered_seed} (0x{res.recovered_seed:04X})')
print(f'Original Seed       : {secret_seed}')
print(f'Pattern Offset in PT: Index {res.sequence_offset}')
print(f"Recovered Plaintext : '{res.recovered_plaintext.decode()}'")

# Formal Cryptographic Verification
is_valid = verify_recovered_result(ciphertext, secret_seed, res.recovered_seed, plaintext, res.recovered_plaintext)
print(f"Cryptographic Verification Status: {'SUCCESS' if is_valid else 'FAILED'}")

## 5. Part B: Predictable Password Reset Token Attack
A simulated web application generates a 16-byte token using MT19937 seeded with the integer Unix timestamp.
The attacker observes the token and tests timestamps within [T - 300, T + 300]:

In [ ]:
from attacks.reset_token_attack import generate_reset_token, predict_reset_token, verify_token_match

# 1. Application generates token
actual_t, token_hex, token_bytes = generate_reset_token()
print(f'Generated Token (Hex): {token_hex}')
print(f'Generation Timestamp : {actual_t}')

# 2. Attacker launches time-window attack (+125s skew)
res = predict_reset_token(token_hex, approx_timestamp=actual_t + 125, window_seconds=300)
print('\n' + res.summary())

# 3. Verification
print(f'Timestamp Recovered Accurately: {res.recovered_timestamp == actual_t}')
print(f'Regenerated Token Verified   : {verify_token_match(res.recovered_timestamp, token_hex)}')

## 6. Mathematical Theory & Cryptanalysis
See `docs/LAB_MANUAL_ASSIGNMENT_10.md` for complete derivations.

### Known Plaintext Keystream Equation
$$C_i = P_i \oplus K_i \implies K_i = P_i \oplus C_i$$

### 16-Bit Keyspace
$$|\mathcal{S}| = 2^{16} = 65,536 \implies \text{Broken in } \approx 0.6 \text{ seconds}$$

### Timestamp Seeding Vulnerability
- Seeding with integer Unix seconds collapses entropy to 1 value/sec.
- Search window $[T-300, T+300] = 601$ candidates ($< 10$ ms attack).